# 05. 리텐션 캠페인 A/B 사전 설계

실제 실험 결과가 아닌 공개 로그 기반 설계다. **3/31 UTC 하루 종료까지**의 활동·구매 이력으로 Recency를 분류하고, **4/1–4/30**의 활동으로 참고 baseline을 계산한다.

이 노트북은 원본 CSV를 스캔하지 않는다. 검증된 `data/processed/recency_rebuild/recency_at_reference.candidate.parquet`, `churn_labels.parquet`, `purchase_users_at_reference.parquet`를 읽는다. 기존 recency 캐시는 덮어쓰지 않는다. 생성 경로는 `python -m src.recency_reference prepare` → `scan` → `python -m src.recency_validation`이며, 후보의 검증·검토 확인 후 사용한다. 구매 캐시 재생성은 별도 `src.ab_design` 작업이다.

## Recency 구간별 4월 활동

대상은 3/31까지 활동이 관측된 사용자다. Recency는 3/31에서 마지막 사전 활동의 UTC 날짜를 뺀 일수다. 구매층은 3/31까지의 구매 경험이며, 활동률은 구간·층별 대상자 중 4월에 한 번 이상 활동한 비율이다. 4월 신규 관측 사용자는 제외한다.

In [1]:
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(root))
import config
import pandas as pd
from IPython.display import Markdown, display
from src.ab_design import summarize_risk, summary_markdown

from src.recency_segments import load_comparison, with_totals, comparison_markdown, check_risk_alignment

strata, risk = load_comparison()
comparison = with_totals(strata)
summary = summarize_risk(risk)
check_risk_alignment(comparison, summary)
comparison.to_csv(config.RECENCY_COMPARISON_PATH, index=False)
display(Markdown(comparison_markdown(comparison)))

| Recency(일) | 구매층 | 대상자 수 | 4월 활동자 수 | 4월 활동률 |
|---|---|---:|---:|---:|
| 0-7 | 전체 | 1,124,507 | 650,816 | 57.88% |
| 0-7 | 사전 구매 | 323,554 | 242,147 | 74.84% |
| 0-7 | 사전 비구매 | 800,953 | 408,669 | 51.02% |
| 8-14 | 전체 | 683,172 | 283,359 | 41.48% |
| 8-14 | 사전 구매 | 157,777 | 92,069 | 58.35% |
| 8-14 | 사전 비구매 | 525,395 | 191,290 | 36.41% |
| 15-29 | 전체 | 2,200,340 | 571,010 | 25.95% |
| 15-29 | 사전 구매 | 322,372 | 136,385 | 42.31% |
| 15-29 | 사전 비구매 | 1,877,968 | 434,625 | 23.14% |
| 30+ (휴면 기준 충족군의 복귀) | 전체 | 9,521,812 | 894,466 | 9.39% |
| 30+ (휴면 기준 충족군의 복귀) | 사전 구매 | 940,661 | 125,186 | 13.31% |
| 30+ (휴면 기준 충족군의 복귀) | 사전 비구매 | 8,581,151 | 769,280 | 8.96% |

**이 표는 관찰 비교이며, 15~29일이 최적의 개입 구간으로 검증된 것은 아니다.** Recency는 방문 빈도·관측 이력과 연관되므로 사전 구매 여부로 층화해도 교란이 남는다. 구간별 활동률 차이를 캠페인의 인과효과나 예상 증분 효과로 해석하지 않는다. 대조 실험이 없어 아래 활동률을 엄밀한 무처치 자연 재방문율로 확정할 수도 없다.

### 15~29일을 설계 대상으로 두는 근거

1. 관측된 4월 활동률은 25.95%로 0~7일(57.88%)·8~14일(41.48%)보다 낮지만 0은 아니다. 이는 개입 효과의 증거가 아니라 대상 선정의 관찰 근거다.
2. 대상 풀은 2,200,340명으로 전체 설계 표본 2,552명보다 크다. 다만 수신 동의·도달 가능 여부가 없어 실제 모집 가능 규모나 층별 검정력 확보를 보장하지 않는다.
3. 기준일 Recency가 30일 미만으로, 사전 휴면 기준(N=30)을 아직 넘기지 않은 구간이다. 30+는 대비용 휴면 기준 충족군이며 4월 활동은 ‘복귀’로 읽는다. 이는 사후 4월 미활동 라벨과 구분한다.

15~29일 선택은 위 관찰과 운영 목적을 결합한 설계 가설이다. 전체 행은 두 구매층의 **분자·분모를 합산**한 값이다. [집계 CSV](../reports/recency_segment_comparison.csv)의 `overall`과 두 층을 함께 더하면 중복 집계가 된다. `baseline`은 0~1 비율이며 아래 설계표와 인원·활동자 수·반올림 전 비율을 대조한다.

## 위험군 baseline과 표본 설계

In [2]:
display(Markdown(summary_markdown(summary)))

| 구분 | 대상자 수 | 4월 재방문자 수 | baseline | 설계 대립값(+5%p) | 군당 표본 |
|---|---:|---:|---:|---:|---:|
| 전체 위험군 | 2,200,340 | 571,010 | 25.95% | 30.95% | 1,276 |
| 기준일 이전 구매 경험군 | 322,372 | 136,385 | 42.31% | 47.31% | 1,552 |
| 기준일 이전 비구매군 | 1,877,968 | 434,625 | 23.14% | 28.14% | 1,195 |

## 입력 시점과 층화

`is_buyer_at_reference`는 관측 시작일부터 3/31 종료까지 구매가 있었는지다. 04의 `is_buyer`는 4월을 포함하므로 이 계산에 사용하지 않는다. 구매 여부만으로 고객을 고가치·저가치라 부르지 않는다.

31.5% vs 10.3%는 사후 구매 구성비이며 사전 예측력·인과효과가 아니다. 위 표의 층별 baseline을 사용한다.

In [3]:
stored = pd.read_csv(config.REPORTS_DIR / "ab_design_summary.csv")
pd.testing.assert_frame_equal(summary, stored, check_exact=False, rtol=1e-12)
assert summary.iloc[1:]["users"].sum() == summary.iloc[0]["users"]
assert summary.iloc[1:]["returned_users"].sum() == summary.iloc[0]["returned_users"]
assert risk["status"].isin(["retained", "churned"]).all()
assert risk["recency_at_ref"].between(15, 29).all()
print("저장 CSV 일치 · 층별 인원/재방문자 합계 일치 · 위험군/라벨 검사 통과")

저장 CSV 일치 · 층별 인원/재방문자 합계 일치 · 위험군/라벨 검사 통과


## 표본과 실행 설계

alpha 0.05, power 0.80, 절대 MDE +5%p, 양측 검정, 처치·대조 1:1이다. 표본 수는 올림하며 설계서 표와 같은 함수를 쓴다.

전체를 1차 판단, 층별을 탐색 분석으로 둔다. 대상 풀의 층 비중을 유지해 추출한 뒤 각 층 내에서 1:1 배정한다. 전체 표본 확보는 각 층의 검정력 확보와 다르며, 층마다 같은 인원을 뽑으면 전체 효과의 모집단 가중치를 다시 정해야 한다.

Primary는 전체 배정 사용자를 분모로 한 30일 재방문율(ITT). Secondary는 구매율·객단가이며 주문 ID가 없어 객단가는 운영 데이터 확보 후 산식을 정한다. Guardrail은 수신거부·비용·마진이며 실행 전에 허용 악화폭을 정한다. MDE는 검정력 설계 조건으로, 자동 성공 기준은 아니다. SRM·중복 배정·관찰 누락을 확인하고 임의 조기 중단을 피한다.

실제 실험을 집행하지 않았으며 수신 동의·비용 정보가 없어 바로 집행 가능한 운영 명세는 아니다. 자세한 산식과 제한은 [A/B 설계서](../docs/ab_test_design.md)를 따른다.

In [4]:
import json
audit = json.loads((config.REPORTS_DIR / "ab_design_validation.json").read_text())
print("기준일:", audit["reference_date"], audit["timezone"])
print("사전 입력 종료(미포함):", audit["feature_end_exclusive"])
print("미래 구매 이력 때문에 잘못 분류됐던 위험군:", f"{audit['reclassified_future_only_buyers']:,}명")
print("그중 4월 재방문자:", f"{audit['reclassified_returned_users']:,}명")

기준일: 2020-03-31 UTC
사전 입력 종료(미포함): 2020-04-01T00:00:00Z
미래 구매 이력 때문에 잘못 분류됐던 위험군: 32,583명
그중 4월 재방문자: 32,583명
